# EXPERIMENT 9 – Improved Transformer-Based LLM Chatbot Using Gradio

**Name:** Mohamed Aashik S  
**Roll Number:** 24BAD072


## 1. Install required libraries

In [ ]:
%pip install -q -U "transformers>=4.45.0" "gradio>=5.0.0" accelerate

## 2. Import libraries and configure device

In [ ]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    DEVICE = "cuda"
    TORCH_DTYPE = torch.float16
else:
    DEVICE = "cpu"
    TORCH_DTYPE = torch.float32

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Data type:", TORCH_DTYPE)

Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda
Data type: torch.float16


## 3. Load the instruction-tuned model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=TORCH_DTYPE,
    device_map="auto"
)

model.eval()
print("Qwen2.5-1.5B-Instruct loaded successfully.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2.5-1.5B-Instruct loaded successfully.


## PART A – Load, tokenize and generate a response

The chat template formats messages using the model's expected instruction structure. This is important for instruction-tuned models.

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful, accurate and friendly AI assistant. "
    "Answer the user's question directly in clear, simple English. "
    "Give relevant facts and explain concepts with examples when useful. "
    "Do not invent information. If you are unsure, clearly say so."
)

sample_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is artificial intelligence? Explain with two examples."}
]

sample_text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=True
)

sample_inputs = tokenizer(
    sample_text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    sample_output = model.generate(
        **sample_inputs,
        max_new_tokens=180,
        do_sample=True,
        temperature=0.3,
        top_p=0.85,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

new_tokens = sample_output[0][sample_inputs["input_ids"].shape[1]:]
sample_answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Sample prompt: What is artificial intelligence?")
print("\nGenerated response:")
print(sample_answer)

Sample prompt: What is artificial intelligence?

Generated response:
Artificial Intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, and self-correction.
Two common examples of AI are:
1. Siri: This is an example of natural language processing (NLP), which allows computers to understand and respond to human speech. Siri uses machine learning algorithms to learn from interactions and improve its responses over time.
2. Autonomous vehicles: These use AI for navigation and decision-making on roads. They can process vast amounts of data from sensors like cameras and radars, as well as maps, to make real-time decisions about speed, direction, and other actions. For instance, if there's a pedestrian crossing ahead, the vehicle might slow down or stop to avoid hitting them.
These examples illustrate how AI enables machines to perform tasks that typically require human intelligence, suc

## PART B – Implement chatbot response generation with conversation history

In [ ]:
def build_messages(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Keep only the most recent conversation turns to control prompt length.
    for item in (history or [])[-6:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in ("user", "assistant") and isinstance(content, str) and content.strip():
                messages.append({"role": role, "content": content})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if user_text:
                messages.append({"role": "user", "content": str(user_text)})
            if assistant_text:
                messages.append({"role": "assistant", "content": str(assistant_text)})

    messages.append({"role": "user", "content": message.strip()})
    return messages


def generate_response(message, history=None, max_new_tokens=220, temperature=0.3):
    if not message or not message.strip():
        return "Please enter a question."

    messages = build_messages(message, history)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=3500
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=max(float(temperature), 0.1),
            top_p=0.85,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return "I couldn't generate a response. Please try rephrasing your question."

    return answer


test_question = "What is machine learning? Explain in simple words."
print("User:", test_question)
print("Assistant:", generate_response(test_question))

User: What is machine learning? Explain in simple words.
Assistant: Machine learning is a type of artificial intelligence that allows computers to learn from data without being explicitly programmed. It involves using algorithms to analyze patterns in large datasets and make predictions or decisions based on those patterns.
For example, imagine you have a dataset containing images of cats and dogs. A machine learning algorithm could be trained to recognize these animals by analyzing thousands of images. Once it has learned this pattern, it can then use that knowledge to classify new images as either cats or dogs even if they haven't seen them before.
The key aspects of machine learning include:
1. **Training**: The system learns from labeled data (data with known outcomes).
2. **Prediction**: It uses what it has learned to predict outcomes for new, unseen data.
3. **Improvement**: As more data becomes available, the model gets better at making accurate predictions.

This technology is 

## PART C – Develop an interactive chatbot using Gradio

The Gradio interface sends the current question and conversation history to the model and displays the generated answer.

In [ ]:
def chat_function(message, history):
    return generate_response(message, history)


demo = gr.ChatInterface(
    fn=chat_function,
    title="Qwen AI Chatbot",
    description="Experiment 9 | Roll No: 24BAD072 | Powered by Qwen2.5-1.5B-Instruct",
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning with an example.",
        "What is a Transformer in deep learning?",
        "What is the difference between AI and machine learning?"
    ],
    chatbot=gr.Chatbot(height=480),
    textbox=gr.Textbox(
        placeholder="Ask a question...",
        container=False
    )
)

print("Gradio chatbot interface created successfully.")

Gradio chatbot interface created successfully.


### Launch the chatbot

In [ ]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae172366beff90d230.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## PART D – Application testing and deployment


In [ ]:
requirements = """torch
transformers>=4.45.0
gradio>=5.0.0
accelerate
"""

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print("requirements.txt created successfully.")

requirements.txt created successfully.
